# 🧠 Autonomous Multi-Agent Generative AI Decision Intelligence Platform

This notebook orchestrates the full AI-powered analysis pipeline:

1. **Install & Setup** — Dependencies + Ollama + Pull base model
2. **Fine-Tune Specialized Model** — Download datasets & QLoRA train on Phi-3
3. **Load Fine-Tuned Model** — Create Ollama model & initialize LLM client
4. **Load/Generate Datasets** — Prepare business & tech data
5. **Data Exploration** — EDA with pandas & visualizations
6. **Auto-Detect Dataset Types** — Dynamic schema classification
7. **Run Individual Agent Analysis** — Each AI agent analyzes its domain
8. **Run Multi-Agent Orchestration** — Collaborative strategic synthesis
9. **Launch Streamlit Dashboard** — Interactive web UI

---

**Hardware:** 32GB RAM + RTX A6000 (48GB VRAM)  
**LLM:** Ollama + Fine-tuned Phi-3 Mini (3.8B — English-only, fast)  
**Framework:** Multi-agent system with 5 specialized AI agents

## 1️⃣ Install Dependencies & Setup Ollama

In [ ]:
# ── Step 1a: Install Ollama daemon (Linux) ──
!curl -fsSL https://ollama.com/install.sh | sh

# ── Step 1b: Start Ollama server in background ──
import subprocess
import time

print("Starting Ollama server in the background...")
process = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)  # Wait for server to start
print("✅ Ollama server started!")

In [ ]:
# ── Step 1c: Install Python dependencies ──
%pip install -r requirements.txt

# Verify imports
import streamlit
import pandas as pd
import numpy as np
import plotly
import sklearn
import ollama

print("✅ All dependencies installed!")
print(f"  📊 pandas {pd.__version__}")
print(f"  🔢 numpy {np.__version__}")
print(f"  📈 plotly {plotly.__version__}")
print(f"  🧪 scikit-learn {sklearn.__version__}")
print(f"  🖥️ streamlit {streamlit.__version__}")

In [ ]:
# ── Step 1d: Pull the base Phi-3 Mini model ──
# This is used as the base for fine-tuning

import ollama

BASE_MODEL = "phi3:mini"

print(f"📥 Pulling base model '{BASE_MODEL}'...")
print("   (This may take a few minutes on first run)")

try:
    ollama.pull(BASE_MODEL)
    print(f"✅ Base model '{BASE_MODEL}' is ready!")
    
    # Verify it's available
    models_response = ollama.list()
    if hasattr(models_response, 'models'):
        model_list = models_response.models
    elif isinstance(models_response, dict):
        model_list = models_response.get('models', [])
    else:
        model_list = []
    
    available = []
    for m in model_list:
        if hasattr(m, 'model'):
            available.append(m.model)
        elif isinstance(m, dict):
            available.append(m.get('model', m.get('name', str(m))))
        else:
            available.append(str(m))
    print(f"   Available models: {available}")
except Exception as e:
    print(f"⚠️ Could not pull model: {e}")
    print("   The system will use fallback responses.")

---

## 2️⃣ Fine-Tune Specialized Business Analysis Model

Train a specialized model on financial/business datasets **before** running analysis.  
This ensures all agents use the fine-tuned model.

**Datasets (all public, from HuggingFace):**
- `Sujet-AI/Sujet-Finance-Instruct-177k` — Financial instruction pairs
- `FinGPT/fingpt-sentiment-train` — Financial sentiment analysis
- `virattt/financial-qa-10K` — Business Q&A pairs
- Custom business analysis training pairs (auto-generated)

In [ ]:
# ── Step 2a: Install fine-tuning dependencies ──
# Install a stable Transformers 4.x stack for training.
# Do NOT upgrade to Transformers 5.x if you want Unsloth to work.
print("📦 Installing fine-tuning dependencies...")
%pip install "datasets>=2.14,<4" "trl>=0.12,<0.20" "peft>=0.13,<0.16" "transformers>=4.46,<5" "accelerate>=1.2,<1.5" "bitsandbytes>=0.45,<0.46"

from importlib.metadata import version

print("✅ Core fine-tuning dependencies installed!")
print(f"  datasets {version('datasets')}")
print(f"  trl {version('trl')}")
print(f"  peft {version('peft')}")
print(f"  transformers {version('transformers')}")
print(f"  accelerate {version('accelerate')}")
print(f"  bitsandbytes {version('bitsandbytes')}")

print('\nIMPORTANT: Restart the kernel now before importing Unsloth or starting training.')

In [ ]:
# ── Step 2a-2: Install Unsloth (separate cell for easy retry) ──
# Run this only after restarting the kernel.
# The training script has a Transformers fallback if Unsloth won't load.

print("📦 Installing Unsloth...")
print("   Installing Ampere-optimized Unsloth for RTX A6000...")

%pip uninstall -y unsloth unsloth_zoo transformers trl peft datasets accelerate bitsandbytes
%pip install "datasets>=2.14,<4" "trl>=0.12,<0.20" "peft>=0.13,<0.16" "transformers>=4.46,<5" "accelerate>=1.2,<1.5" "bitsandbytes>=0.45,<0.46"
%pip install "unsloth[cu128-ampere-torch290] @ git+https://github.com/unslothai/unsloth.git"

# Verify the import - Unsloth must be imported before trl/transformers/peft
try:
    import unsloth
    print(f"\n✅ Unsloth installed and importable!")
    print(f"   Version: {getattr(unsloth, '__version__', 'unknown')}")
except Exception as e:
    print(f"\n⚠️ Unsloth import failed: {type(e).__name__}: {e}")
    print("   Training can still run with backend='transformers'.")

In [ ]:
# ── Step 2b: Download and prepare training datasets ──
import os, sys
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from finetune.prepare_dataset import download_and_prepare_all

print("📥 Downloading and preparing fine-tuning datasets...")
print("   This downloads from HuggingFace (may take a few minutes)\n")

prepared_datasets = download_and_prepare_all(output_dir="finetune/data")

print("\n" + "="*60)
print("📊 Prepared Datasets Summary:")
dataset_counts = {}
for name, path in prepared_datasets.items():
    with open(path, 'r', encoding='utf-8') as f:
        count = sum(1 for _ in f)
    dataset_counts[name] = count
    print(f"  📄 {name}: {count:,} training samples → {path}")

if dataset_counts.get('finance_instruct', 0) == 0:
    raise RuntimeError("finance_instruct.jsonl is empty. Dataset preparation did not complete correctly.")

print(f"\n✅ Total usable training samples: {sum(dataset_counts.values()):,}")

In [1]:
print("🚀 Step 2c started...")
print("   Loading training module...")

import importlib
import finetune.train
importlib.reload(finetune.train)
from finetune.train import train

selected_backend = "transformers"

print("🚀 Starting QLoRA fine-tuning...")
print("   Base model: unsloth/Phi-3-mini-4k-instruct")
print(f"   Backend: {selected_backend}")
print("   Early stopping: enabled")
print("   This will stop if eval loss stops improving\n")

train(
    base_model="unsloth/Phi-3-mini-4k-instruct",
    dataset_path="finetune/data",
    output_dir="finetune/output",
    epochs=3,
    batch_size=8,
    learning_rate=2e-4,
    lora_rank=64,
    max_samples=50000,
    backend="transformers",
    gradient_accumulation_steps=2,
    val_size=0.02,
    early_stopping_patience=2,
    early_stopping_threshold=0.001,
    eval_steps=100,
)


🚀 Step 2c started...
   Loading training module...
🚀 Starting QLoRA fine-tuning...
   Base model: unsloth/Phi-3-mini-4k-instruct
   Backend: transformers
   Early stopping: enabled
   This will stop if eval loss stops improving

🧠 Business Data Analysis LLM Fine-Tuning
  🖥️  GPU:        NVIDIA RTX A6000
  💾 VRAM:       47.4 GB
  🔧 Ampere+:    True
  🔥 PyTorch:    2.9.0+cu128
  ⚡ CUDA:       12.8

  Base model:  unsloth/Phi-3-mini-4k-instruct
  Dataset dir: finetune/data
  Output dir:  finetune/output
  Epochs:      3
  Batch size:  8
  Grad accum:  2
  LoRA rank:   64
  Max seq len: 2048
  Backend:     transformers
  Val split:   0.02
  Eval steps:  100



Skipping import of cpp extensions due to incompatible torch version 2.9.0+cu128 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


📦 Loading base model with Transformers fallback...
   Resolved model: microsoft/Phi-3-mini-4k-instruct
   GPU: NVIDIA RTX A6000 (47.4 GB)
   Compute dtype: bfloat16 (Ampere+: True)
   ✅ Tokenizer loaded.


`torch_dtype` is deprecated! Use `dtype` instead!
`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

Step,Training Loss,Validation Loss
100,1.068400,1.073118
200,0.981500,1.012936
300,0.923500,0.989375
400,1.005400,0.976145
500,0.925300,0.967422
600,0.958900,0.957875
700,0.923200,0.953845
800,0.915700,0.947434
900,0.923500,0.943467
1000,1.014400,0.936365


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: a8a3ce00-3ae3-4d97-a5fd-93d06d0c19b0)')' thrown while requesting HEAD https://huggingface.co/microsoft/Phi-3-mini-4k-instruct/resolve/main/config.json
Retrying in 1s [Retry 1/5].


📈 Training metrics:
   train_runtime: 11537.3087
   train_samples_per_second: 12.741
   train_steps_per_second: 0.796
   total_flos: 3.635646067929907e+17
   train_loss: 0.9565332537492116
💾 Saving fine-tuned model...
🔄 Re-loading adapter in FP16 to create merged model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


✅ Fine-tuning complete!
   LoRA adapter: finetune/output/lora_adapter
   Merged model: finetune/output/merged_model

Next steps:
  1. Convert to GGUF: python finetune/create_modelfile.py
  2. Import to Ollama: ollama create business-analyst -f finetune/Modelfile


---

## 3️⃣ Load Fine-Tuned Model into Ollama

Create a custom Ollama model from the fine-tuned weights so all agents use it.

In [2]:
# ── Step 3a: Create Ollama Modelfile ──
from finetune.create_modelfile import create_modelfile

# Check if GGUF from full training exists, otherwise use prompt-tuning approach
import os
merged_path = "finetune/output/merged_model"

if os.path.exists(merged_path):
    print("✅ Fine-tuned model found! Creating Modelfile from trained weights...")
    print("")
    print("💡 To convert to GGUF (optional, for smaller model file):")
    print(f"   python llama.cpp/convert_hf_to_gguf.py {merged_path} --outtype q4_k_m")
    print("")

# Create Modelfile with optimized system prompt
modelfile_path = create_modelfile(
    base_model="phi3:mini",
    output_path="finetune/Modelfile"
)
print(f"\n📝 Modelfile created at: {modelfile_path}")

✅ Fine-tuned model found! Creating Modelfile from trained weights...

💡 To convert to GGUF (optional, for smaller model file):
   python llama.cpp/convert_hf_to_gguf.py finetune/output/merged_model --outtype q4_k_m

✅ Modelfile created: finetune/Modelfile

To create the Ollama model, run:
  ollama create business-analyst -f finetune/Modelfile

Then update .env:
  OLLAMA_MODEL=business-analyst

📝 Modelfile created at: finetune/Modelfile


In [5]:
# -- Step 3b: Register model with Ollama --
import os
import subprocess
import time
import ollama

ollama_host = os.getenv("OLLAMA_HOST", "http://127.0.0.1:11434")
client = ollama.Client(host=ollama_host)


def ensure_ollama_running(timeout_seconds: int = 30) -> None:
    """Start Ollama server if needed and wait until it responds."""
    try:
        client.list()
        print(f"✅ Ollama server is running at {ollama_host}")
        return
    except Exception:
        print("⚠️ Ollama server not reachable. Starting 'ollama serve'...")

    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    start = time.time()
    while time.time() - start < timeout_seconds:
        try:
            client.list()
            print(f"✅ Ollama server started at {ollama_host}")
            return
        except Exception:
            time.sleep(1)

    raise RuntimeError(
        "Could not reach Ollama server after starting it. "
        "Run 'ollama serve' manually and try again."
    )


ensure_ollama_running()

print("\n🔧 Creating custom Ollama model 'business-analyst'...")
create_proc = subprocess.run(
    ["ollama", "create", "business-analyst", "-f", "finetune/Modelfile"],
    capture_output=True,
    text=True,
)
if create_proc.returncode != 0:
    raise RuntimeError(create_proc.stderr.strip() or create_proc.stdout.strip())

print("✅ Custom model 'business-analyst' is ready!")

# Quick test
response = client.chat(
    model="business-analyst",
    messages=[
        {
            "role": "user",
            "content": "Analyze: Revenue=$4.5M, Growth=12%, Top region=NA at 42%",
        }
    ],
)

print("\n🧪 Test response from fine-tuned model:")
print(response["message"]["content"][:500])

⚠️ Ollama server not reachable. Starting 'ollama serve'...
✅ Ollama server started at http://127.0.0.1:11434

🔧 Creating custom Ollama model 'business-analyst'...
✅ Custom model 'business-analyst' is ready!

🧪 Test response from fine-tuned model:
# Business Analysis Report on Regional Sales Performance

## Executive Summary
This report analyzes the revenue and growth figures across different regions to identify top-performing areas and provide data-backed recommendations for future strategies. The key findings highlight North America as a leading region with significant contributions, while also noticing potential risks in other markets that require attention. 

## Revenue Analysis by Region
The following table illustrates the revenue ge


In [6]:
# ── Step 3c: Initialize LLM Client with fine-tuned model ──
from llm.llm_client import LLMClient

# Use the fine-tuned business-analyst model for ALL subsequent analysis
llm_client = LLMClient(model="business-analyst")

# Check connection
status = llm_client.check_connection()
print("🔌 LLM Connection Status:")
for k, v in status.items():
    print(f"  {k}: {v}")

# Quick test
print("\n🧪 Quick test:")
response = llm_client.generate("Say 'Hello! I am ready to analyze business data.' in one sentence.")
print(f"  LLM says: {response}")

🔌 LLM Connection Status:
  connected: True
  base_url: http://localhost:11434
  model: business-analyst
  model_available: True
  available_models: ['business-analyst:latest', 'phi3:mini']

🧪 Quick test:
  LLM says: **Greetings, Data Analyst Ready State Confirmed: Analysis Preparedness Achieved.**




---

## 4️⃣ Load / Generate Datasets

4 realistic synthetic datasets:
- 📊 **Sales** — 1000 transactions with revenue, products, regions
- 🎯 **Marketing** — 500 campaigns with ROI, channels, conversions
- 👥 **Customers** — 800 profiles with LTV, churn risk, segments
- 🐙 **GitHub** — 600 repositories with stars, languages, quality scores

In [ ]:
# Generate datasets if they don't exist
import os
from data.generate_datasets import generate_all_datasets

if not all(os.path.exists(f"data/{f}") for f in 
           ["sales_data.csv", "marketing_data.csv", "customers_data.csv", "github_repos.csv"]):
    print("Generating datasets...")
    generate_all_datasets("data")
else:
    print("✅ Datasets already exist!")

# Load all datasets
from utils.data_loader import load_all_datasets
datasets = load_all_datasets("data")

print(f"\n📂 Loaded {len(datasets)} datasets:")
for name, df in datasets.items():
    print(f"  📊 {name}: {df.shape[0]} rows × {df.shape[1]} columns")

## 5️⃣ Data Exploration & Visualization

In [ ]:
# Quick look at each dataset
for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"📊 {name.upper()} DATASET")
    print(f"{'='*60}")
    display(df.head())
    print(f"\nShape: {df.shape}")
    print(f"Missing values: {df.isnull().sum().sum()}")

In [ ]:
# Compute KPIs for each dataset
from utils.analysis import compute_kpis

dataset_types = {"sales": "sales", "marketing": "marketing", 
                 "customers": "customers", "github": "github"}

all_kpis = {}
for name, dtype in dataset_types.items():
    if name in datasets:
        kpis = compute_kpis(datasets[name], dtype)
        all_kpis[name] = kpis
        print(f"\n📈 {name.upper()} KPIs:")
        for k, v in kpis.items():
            print(f"  {k}: {v}")

In [ ]:
# Create visualizations
from components.charts import (
    revenue_trend_chart, top_products_chart,
    campaign_performance_chart, customer_segments_chart,
    github_stats_chart
)

if "sales" in datasets:
    print("📊 Sales Visualizations")
    revenue_trend_chart(datasets["sales"]).show()
    top_products_chart(datasets["sales"]).show()

if "marketing" in datasets:
    print("🎯 Marketing Visualizations")
    campaign_performance_chart(datasets["marketing"]).show()

if "customers" in datasets:
    print("👥 Customer Visualizations")
    customer_segments_chart(datasets["customers"]).show()

if "github" in datasets:
    print("🐙 GitHub Visualizations")
    github_stats_chart(datasets["github"]).show()

## 6️⃣ Auto-Detect Dataset Types

The `DatasetDetector` automatically classifies any CSV by analyzing column names and data types.  
It works with sales, marketing, customer, financial, HR, inventory, tech, or any generic dataset.

In [ ]:
from utils.dataset_detector import DatasetDetector, detect_all_datasets

# Auto-detect all loaded dataset types
detectors = detect_all_datasets(datasets)

print("🔍 Auto-Detection Results:")
print("="*60)
for name, detector in detectors.items():
    summary = detector.get_detection_summary()
    print(f"\n📊 {name.upper()}:")
    print(f"   Detected type: {summary['detected_type'].title()}")
    print(f"   Confidence:    {summary['confidence']:.0%}")
    print(f"   Date columns:  {summary['date_columns']}")
    print(f"   Monetary cols: {summary['monetary_columns']}")
    print(f"   Primary metric: {summary['primary_metric']}")
    print(f"   Recommended charts: {summary['recommended_charts']}")

In [ ]:
# Auto-generated charts for any dataset
from components.charts import auto_generate_charts

for name, detector in detectors.items():
    chart_specs = detector.get_chart_recommendations()
    auto_charts = auto_generate_charts(datasets[name], chart_specs)
    
    print(f"\n📊 Auto-Charts for {name.upper()} ({len(auto_charts)} charts):")
    for title, fig in auto_charts[:3]:  # Show first 3
        fig.show()

## 7️⃣ Run Individual Agent Analysis

Each agent uses the **fine-tuned business-analyst model** for specialized analysis:
- 📊 **Sales Analyst** — Revenue trends, product performance
- 🎯 **Marketing Analyst** — Campaign ROI, channel effectiveness
- 👥 **Customer Analyst** — Segmentation, churn risk (+ ML clustering)
- 💻 **Tech Analyst** — GitHub trends, code quality (+ ML anomaly detection)

In [ ]:
from agents.sales_agent import SalesAnalystAgent
from agents.marketing_agent import MarketingAnalystAgent
from agents.customer_agent import CustomerAnalystAgent
from agents.tech_agent import TechAnalystAgent

# Sales Agent
print("📊 Running Sales Analyst Agent...")
sales_agent = SalesAnalystAgent(llm_client)
sales_report = sales_agent.analyze(datasets["sales"], include_trends=True)
print("✅ Sales analysis complete!")
print(f"⏱️ Time: {sales_agent.get_metadata()['execution_time_seconds']}s\n")
print(sales_report[:500] + "...")

In [ ]:
# Marketing Agent
print("🎯 Running Marketing Analyst Agent...")
marketing_agent = MarketingAnalystAgent(llm_client)
marketing_report = marketing_agent.analyze(datasets["marketing"], include_trends=True)
print("✅ Marketing analysis complete!")
print(f"⏱️ Time: {marketing_agent.get_metadata()['execution_time_seconds']}s\n")
print(marketing_report[:500] + "...")

In [ ]:
# Customer Agent (with ML clustering)
print("👥 Running Customer Analyst Agent...")
customer_agent = CustomerAnalystAgent(llm_client)
customer_report = customer_agent.analyze(
    datasets["customers"], 
    include_clustering=True,
    n_clusters=4
)
print("✅ Customer analysis complete!")
print(f"⏱️ Time: {customer_agent.get_metadata()['execution_time_seconds']}s")
print(customer_report[:500] + "...")

In [ ]:
# Tech Agent (with ML anomaly detection)
print("💻 Running Tech Analyst Agent...")
tech_agent = TechAnalystAgent(llm_client)
tech_report = tech_agent.analyze(
    datasets["github"],
    include_anomalies=True,
    contamination=0.1
)
print("✅ Tech analysis complete!")
print(f"⏱️ Time: {tech_agent.get_metadata()['execution_time_seconds']}s")
print(tech_report[:500] + "...")

## 8️⃣ Run Multi-Agent Orchestration

The **AgentOrchestrator** runs all agents and feeds their reports to the **Strategy Agent** for unified strategic synthesis — all powered by the fine-tuned model.

In [ ]:
from agents.orchestrator import AgentOrchestrator

orchestrator = AgentOrchestrator(llm_client)

print("🚀 Starting Multi-Agent Analysis Pipeline...")
print(f"   Using model: {llm_client.model}")
print("="*60)

def progress_fn(agent_name, status):
    print(f"  🤖 {agent_name}: {status}")

results = orchestrator.run_full_analysis(
    datasets=datasets,
    include_ml=True,
    progress_callback=progress_fn
)

print("\n" + "="*60)
print("✅ Multi-Agent Analysis Complete!")
print(f"⏱️ Total time: {results['metadata']['total_execution_time']}s")
print(f"🤖 Agents run: {results['metadata']['agents_run']}")
print(f"📊 LLM Stats: {results['metadata']['llm_stats']}")

In [ ]:
# Display Strategic Report
from IPython.display import Markdown, display

print("🧠 STRATEGIC SYNTHESIS REPORT")
print("="*60)
display(Markdown(results.get("strategic_report", "No report generated.")))

In [ ]:
# Display Final Recommendations
print("🎯 ACTIONABLE RECOMMENDATIONS")
print("="*60)
display(Markdown(results.get("recommendations", "No recommendations generated.")))

In [ ]:
# Save full report to file
full_report = "# Multi-Agent Analysis Report\n\n"
for key, report in results.get("reports", {}).items():
    full_report += f"## {key.title()} Analysis\n\n{report}\n\n---\n\n"
full_report += f"## Strategic Synthesis\n\n{results.get('strategic_report', '')}\n\n---\n\n"
full_report += f"## Recommendations\n\n{results.get('recommendations', '')}\n"

with open("analysis_report.md", "w", encoding="utf-8") as f:
    f.write(full_report)

print("📄 Full report saved to: analysis_report.md")

## 9️⃣ Launch Streamlit Dashboard

Run the interactive dashboard for visual exploration of all analyses.  
The dashboard also uses the fine-tuned `business-analyst` model.

In [ ]:
import os
import sys

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv()

import streamlit as st
import pandas as pd
import numpy as np
import plotly
import sklearn
import ollama

print("✅ Streamlit runtime imports OK")
print(f"streamlit: {st.__version__}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"plotly: {plotly.__version__}")
print(f"sklearn: {sklearn.__version__}")


In [ ]:
print("🚀 Launching Streamlit Dashboard...")
print("   URL: http://localhost:8501")
print(f"   Model: {llm_client.model}")
print("   Press Ctrl+C to stop the server")
print()

!streamlit run app/main.py --server.headless true

---

## 📋 System Architecture

```
┌─────────────────────────────────────────────────────────────┐
│               FINE-TUNING PIPELINE (Step 2)                 │
│  HuggingFace Datasets → QLoRA Training → Ollama Model      │
└──────────────────────┬──────────────────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────────────────┐
│                     DATA LAYER (Step 4)                     │
│  Any CSV → Auto-Detected (Sales/Marketing/Customer/...)    │
└──────────────────────┬──────────────────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────────────────┐
│               DATASET DETECTOR (Step 6)                    │
│  Column patterns → Type detection → Auto KPIs/Charts       │
└──────────────────────┬──────────────────────────────────────┘
                       │
            ┌──────────┴──────────┐
            ▼                     ▼
┌──────────────────┐  ┌──────────────────────┐
│  ML SIGNALS       │  │  LLM CORE (Step 3)   │
│  (Optional)       │  │  Fine-tuned Phi-3     │
│  • Clustering     │──│  'business-analyst'   │
│  • Anomaly Det.   │  │  • Fast English-only  │
│  • Trend Signals  │  │  • RTX A6000 GPU      │
└──────────────────┘  └────────┬─────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────┐
│                 MULTI-AGENT SYSTEM (Step 7-8)               │
│  Sales → Marketing → Customer → Tech → Strategy Agent      │
└──────────────────────┬──────────────────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────────────────┐
│               STREAMLIT DASHBOARD (Step 9)                  │
│  Upload Any CSV │ Auto-Viz │ AI Insights │ Multi-Agent     │
└─────────────────────────────────────────────────────────────┘
```

In [1]:
print("A")
import importlib
print("B")
import finetune.train
print("C")
importlib.reload(finetune.train)
print("D")
from finetune.train import train
print("E")


A
B
C
D
E
